# Fáze 0 — Pilot

Ověření technického pipeline (stažení, metriky, polar plot) na malém vzorku, viz `MasterOutline.md` §4. **Stahování spouštěj sám** — jednou stažená obec se cachuje do `data/raw/graphs/*.graphml` a příště se už nestahuje, takže tuhle buňku je bezpečné spouštět opakovaně / navázat po přerušení.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd

from src.config import DATA_PROCESSED
from src.viz import set_style, save_fig
from src.streets.registry import find_municipality
from src.streets.osm import download_graph
from src.streets.metrics import extract_metrics, build_metrics_table, prefetch
from src.streets.viz import plot_polar_grid

set_style()

## Vzorek

`sample_seed.csv` = tvůj seznam měst + Havířov, disambiguovaný přes RÚIAN okres (viz `src/streets/registry.py`). Pro pilot bereme malý kontrastní podvzorek.

In [ ]:
sample = pd.read_csv(DATA_PROCESSED / "sample_seed.csv", encoding="utf-8")

pilot_names = ["Praha", "Brno", "Zlín", "Havířov", "Loštice", "Štíty", "Litomyšl"]
pilot = sample[sample["name"].isin(pilot_names)]
pilot

## Stažení + metriky

`network_type="drive"` — viz sekce srovnání níže pro `drive` vs. `all` a proč jsme u `drive` nakonec zůstali (`MasterOutline.md` §5). Binning bearingu je pořád otevřené rozhodnutí, rozhodni po vizuální kontrole níže.

Grafy se cachují do `data/raw/graphs/`, metriky (počítání betweenness centrality umí být na velkém městě pomalé) do `data/processed/pilot_metrics.csv` — bezpečné spouštět tuhle buňku opakovaně, přepočítá se jen to, co tam ještě není.

In [ ]:
network_type = "drive"  # rozhodnuto (viz sekce srovnání níže + MasterOutline.md §5)
metrics_path = DATA_PROCESSED / "pilot_metrics.csv"

# metriky se počítají jen pro sídla, co ještě nejsou v metrics_path (viz src/streets/metrics.py::build_metrics_table) —
# opakované spuštění téhle buňky nic nepočítá znovu, jen dotáhne případná nová sídla
all_metrics = build_metrics_table(pilot, network_type=network_type, csv_path=metrics_path)
metrics_df = all_metrics[all_metrics["network_type"] == network_type].reset_index(drop=True)

graphs = {row["name"]: download_graph(row, network_type=network_type) for _, row in pilot.iterrows()}
metrics_df

## Polar ploty orientace

Vizuální kontrola: dávají rozdíly mezi historickým jádrem, plánovaným sídlištěm (Havířov) a menšími sídly smysl?

In [ ]:
fig = plot_polar_grid(graphs, num_bins=36, ncols=4)
save_fig(fig, "00_pilot_polar_grid")

## Srovnání `network_type`: `drive` vs. `all` (historický záznam rozhodnutí)

`"drive"` zahrnuje jen silnice sjízdné autem; `"all"` přidá i chodníky, cyklostezky, pěší zóny apod. Siluety **Loštic a Štítů** (malá sídla, pár tisíc obyvatel) byly na `"drive"` nápadně řídké — podezření, že `"drive"` u tak malých obcí zachytí jen hrstku hlavních silnic. To vedlo k prozatímnímu přechodu na `"all"` pro celý vzorek (2026-09-01).

**Rozhodnutí zvráceno (2026-09-08):** `"all"` na velkém městě (Brno) opakovaně selhávalo — Overpass tak velký dotaz tiše ořezal (0 hran místo chyby), a to i po zvýšení timeoutu/paměti a přidání retry logiky. Riziko nespolehlivého stahování u velkých měst převážilo nad benefitem hustší sítě u malých sídel, takže zůstáváme u **`"drive"` v celém projektu**. Řidší síť u malých sídel je otevřené omezení do budoucna — případně řešit přes lokální `.osm.pbf` (viz `MasterOutline.md` §5/6), ne přes `"all"` na veřejném Overpassu.

In [ ]:
compare_names = ["Loštice", "Štíty", "Zlín"]
compare_rows = sample[sample["name"].isin(compare_names)]

compare_metrics_df = (
    all_metrics[all_metrics["name"].isin(compare_names)]
    .sort_values(["name", "network_type"])
    .reset_index(drop=True)
)

compare_graphs = {
    f"{row['name']} ({nt})": download_graph(row, network_type=nt)
    for _, row in compare_rows.iterrows()
    for nt in ["drive", "all"]
}

compare_metrics_df

In [ ]:
fig = plot_polar_grid(compare_graphs, num_bins=36, ncols=2)
save_fig(fig, "00_pilot_drive_vs_all")

**Rozhodnuto:** `network_type="drive"` pro celý projekt (viz sekce výše a `MasterOutline.md` §5). `01_full_sample.ipynb` už s tím počítá.